# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors
This notebook provides a step-by-step guide to loading and exploring the FAIR^2 dataset using the `mlcroissant` library. You will learn how to load metadata, inspect record sets, extract tabular data, process records, and visualize trends with dynamic field and ID usage via the Croissant schema.

### Dataset Source
The dataset is described via the Croissant schema at the following URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` and visualization libraries are installed
!pip install -q mlcroissant matplotlib seaborn

## 1. Data Loading
Load dataset metadata from the FAIR^2 Croissant schema.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Define the dataset schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}\n\n{metadata.description}")

## 2. Data Overview
Explore the structure of the dataset: record sets, fields, and their `@id`s.

In [ ]:
# Display available record sets and their @id
record_sets_info = []
for record_set in metadata.record_sets:
    rec_id = record_set.id
    rec_name = getattr(record_set, 'name', '(No name)')
    print(f"Record set @id: {rec_id}\n  Name: {rec_name}")
    record_sets_info.append({'@id': rec_id, 'name': rec_name})
    # Display fields in the record set
    print("  Fields:")
    for field in getattr(record_set, 'fields', []):
        f_id = field.id
        f_name = getattr(field, 'name', '(No name)')
        print(f"    Field @id: {f_id} (name: {f_name})")
    print("\n")
# Save a list of record set @ids for later use
record_sets_ids = [info['@id'] for info in record_sets_info]

## 3. Data Extraction
Extract data from a selected record set and inspect the first few records as a DataFrame. We will use the first available record set for demonstration.

In [ ]:
# Choose the first record set
if not record_sets_ids:
    raise ValueError("No record sets found in metadata.")
record_set_id = record_sets_ids[0]
print(f"Using record set @id: {record_set_id}\n")

# Extract data and load as DataFrame
records = list(dataset.records(record_set=record_set_id))
df = pd.DataFrame(records)
print("Field @ids (column names):")
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Let's process the dataset by filtering on a numeric field, normalizing it, and grouping analysis. We will choose the first numeric-typed field in the record set for illustration. All field references use `@id`.

In [ ]:
# Identify a numeric field by inspecting the field metadata
fields = None
for rs in metadata.record_sets:
    if rs.id == record_set_id:
        fields = rs.fields
        break
if fields is None:
    raise ValueError("No fields found for the selected record set.")

numeric_field_id = None
for f in fields:
    dt = getattr(f, 'data_type', None)
    if dt in ["Integer", "Float", "Number"]:
        numeric_field_id = f.id
        print(f"Selected numeric field: {numeric_field_id} (type: {dt})")
        break
if not numeric_field_id:
    raise ValueError("No numeric field found for EDA.")

# Check if the field contains non-numeric data due to missingness etc. Try to coerce
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

# Set a threshold for filtering
threshold = df[numeric_field_id].mean() if df[numeric_field_id].notna().any() else 0
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
print(filtered_df.head())

# Normalize the field
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"\nNormalized '{numeric_field_id}' for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by first categorical/string field
group_field_id = None
for f in fields:
    dt = getattr(f, 'data_type', None)
    if dt in ["Text", "String"] and f.id != numeric_field_id:
        group_field_id = f.id
        print(f"Selected group field: {group_field_id}")
        break

if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nMean of '{numeric_field_id}' grouped by '{group_field_id}':")
    print(grouped_df.head())

## 5. Visualization
Visualize the distribution of the selected numeric field and its relationship with the group field (if available).

In [ ]:
# Histogram of the numeric field
plt.figure(figsize=(7, 4))
sns.histplot(df[numeric_field_id].dropna(), kde=True, color='cornflowerblue')
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# Boxplot by group field if available
if group_field_id and group_field_id in df.columns:
    plt.figure(figsize=(10,5))
    sns.boxplot(x=df[group_field_id], y=df[numeric_field_id], color='dodgerblue')
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated end-to-end loading, extraction, and initial exploration of the FAIR^2 dataset using `mlcroissant`, referencing all entities via their `@id`s. We provided dynamic data field handling, cleaned numerical data, performed basic normalization, and visualized interesting patterns. For detailed biomedical or statistical inference, further domain-specific analysis is recommended.